# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## First - let's talk about the Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!

### We will start by calling OpenAI again - but don't worry non-OpenAI people, your time is coming!


In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## Do you know what an Endpoint is?

If not, please review the Technical Foundations guide in the guides folder

And, here is an endpoint that might interest you...

In [4]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [5]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-E4q4eZCiavmFiLtXyokJSSxNYRypz',
 'object': 'chat.completion',
 'created': 1784822192,
 'model': 'gpt-5-nano-2025-08-07',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Fun fact: Bananas are technically berries, but strawberries aren’t. Botanically, a berry is a fruit from a single ovary with seeds inside the flesh. Bananas fit that, while strawberries are aggregate fruits with the tiny seeds on the outside (the true fruits are the little seeds). Want another fun fact?',
    'refusal': None,
    'annotations': []},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 11,
  'completion_tokens': 777,
  'total_tokens': 788,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 704,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': None}

In [6]:
response.json()["choices"][0]["message"]["content"]

'Fun fact: Bananas are technically berries, but strawberries aren’t. Botanically, a berry is a fruit from a single ovary with seeds inside the flesh. Bananas fit that, while strawberries are aggregate fruits with the tiny seeds on the outside (the true fruits are the little seeds). Want another fun fact?'

# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!


In [9]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content



'Fun fact: A group of flamingos is called a flamboyance. Flamingos get their pink color from carotenoid pigments in their diet.'

## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, google made one here: https://generativelanguage.googleapis.com/v1beta/openai/

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

If you're confused, please review Guide 9 in the Guides folder!

And now let's try it!

## THIS IS OPTIONAL - but if you wish to try out Google Gemini, please visit:

https://aistudio.google.com/

And set up your API key at

https://aistudio.google.com/api-keys

And then add your key to the `.env` file, being sure to Save the .env file after you change it:

`GOOGLE_API_KEY=AIz...`


In [10]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
else:
    print("API key found and looks good so far!")



No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini


In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [12]:
requests.get("http://localhost:11434").content

b'Ollama is running'

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [13]:
!ollama pull llama3.2

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [14]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [15]:
# Get a fun fact

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

"Did you know that honey never spoils? Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and are still edible! Honey's unique properties make it virtually immortal - its low moisture content and acidic pH prevent the growth of bacteria and other microorganisms that can cause spoilage. Isn't that sweet?"

In [16]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 705 KB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 1.6 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 5.2 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏  11 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏  14 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   2% ▕                  ▏  20 MB/1.1 GB                  pulling man

In [17]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'\n\nSure! Here\'s a fun fact for you: **"Water is the most polar molecule in the universe."** It acts as one of the best dissuaders of heat, and its high polarity can be seen to spread out light like water droplets through reflection.'

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [21]:
from IPython.display import Markdown, display
from openai import OpenAI

requests.get("http://localhost:11434").content

!ollama pull llama3.2

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

# Step 1: Create your prompts

system_prompt = """
You are a helpful assistant that checks my emails and collect from them a short summary of the email
"""

user_prompt = """
    Here are my emails:
"""

# Step 2: Make the messages list
prompt = user_prompt + """Sample Email for Job Application with Resume: How to Craft a Professional and Effective Email
Applying for a job via email has become a standard practice in today's digital age. Whether responding to a job posting, sending a speculative application, or following up on a referral, knowing how to craft a professional job application email with your resume attached is crucial. This article provides a step-by-step guide, along with sample email templates, to help you create a compelling and effective job application email.


Why Is Your Job Application Email Important?
Your job application email is your first point of contact with the employer. It sets the tone for your professional communication and creates a lasting impression. A well-crafted email can significantly enhance your chances of getting noticed and being invited for an interview.


Key Components of a Job Application Email
When writing a job application email, make sure it includes the following essential elements:

Clear Subject Line: Your subject line should be concise and informative. It should include your name, the job title, and a reference number (if applicable).
Example: Application for Marketing Manager Position - John Doe

Professional Greeting: Address the email to the hiring manager or recruiter by name if possible. Avoid generic greetings like "To whom it may concern."
Example: Dear Ms. Smith,

Introduction: Start with a brief introduction, mentioning how you found the job opening and why you are applying. Highlight your interest in the company and the role.

Body of the Email: Provide a summary of your relevant experience, skills, and qualifications. Keep it concise but impactful, aligning your strengths with the job requirements.

Call to Action: Express your enthusiasm for the opportunity and include a call to action, such as requesting an interview or asking for a follow-up.

Closing Statement: End with a professional closing statement, offering to provide more information if needed.

Signature: Include your full name, phone number, LinkedIn profile (if applicable), and any other relevant contact information.

Attachments: Make sure to attach your resume and any other required documents in the specified format (e.g., PDF or Word document). Label the attachments professionally.
Example: John_Doe_Resume.pdf


Sample Email for Job Application with Resume
Sample 1: Applying for a Job After Seeing a Job Posting
Subject: Application for Software Developer Position - Jane Smith

Dear Mr. Johnson,

I am writing to express my interest in the Software Developer position at [Company Name], as advertised on [Job Board/Company Website]. With a Bachelor’s degree in Computer Science and over three years of experience in developing dynamic web applications, I am confident in my ability to contribute effectively to your team.

In my previous role at [Previous Company], I successfully led a project that reduced processing time by 30%, enhancing operational efficiency. I am skilled in JavaScript, React, and Python, and I am particularly drawn to this position due to [Company Name]’s innovative approach to technology.

I have attached my resume for your review. I would appreciate the opportunity to discuss my application with you further and explore how my skills align with your needs. Thank you for considering my application.

Sincerely,
Jane Smith
[Phone Number]
[LinkedIn Profile]


Sample 2: Speculative Job Application Email
Subject: Enquiry About Potential Job Opportunities - Michael Brown

Dear [Hiring Manager’s Name],

I hope this email finds you well. My name is Michael Brown, and I am a [Your Profession] with [Number] years of experience in [Industry/Field]. I am reaching out to express my interest in potential job opportunities at [Company Name].

With a strong background in [Key Skills/Expertise], I am keen to bring my expertise to your team. I am particularly impressed by [Company Name]’s vision and commitment to [Relevant Industry/Project], and I would love to contribute to your ongoing success.

I have attached my resume for your consideration. I would be grateful if you could keep me in mind for any current or future roles that align with my experience. Thank you for your time and consideration.

Best regards,
Michael Brown
[Phone Number]
[LinkedIn Profile]


Sample 3: Referral-Based Job Application Email
Subject: Referral by [Referrer’s Name] - Application for Marketing Specialist

Dear [Hiring Manager’s Name],

I was referred to this position by [Referrer’s Name], who mentioned that your team is looking for a Marketing Specialist. With a robust background in digital marketing and a passion for driving brand growth, I am excited about the opportunity to contribute to [Company Name].

At [Previous Company], I successfully managed campaigns that increased engagement by 50% within six months. I am confident that my expertise in SEO, content strategy, and analytics can add value to your team.

Please find my resume attached for your review. I look forward to the possibility of discussing my application with you. Thank you for your time and consideration.

Warm regards,
[Your Name]
[Phone Number]
[LinkedIn Profile]


Tips for Writing a Professional Job Application Email
Personalize Your Email: Whenever possible, address the email to a specific person.
Be Concise and Focused: Keep the email short and to the point, highlighting your qualifications.
Follow Instructions: If the job posting includes specific instructions for applying, follow them precisely.
Proofread Your Email: Double-check for grammar, spelling, and formatting errors.
Use a Professional Email Address: Make sure your email address is professional and straightforward.

Conclusion
Writing a job application email with a resume attachment may seem straightforward, but it requires careful attention to detail. By following the structure and tips provided in this guide, you can increase your chances of making a positive impression on potential employers. Use the sample emails as a template and tailor them to suit your specific circumstances. Good luck with your job search!"""

# message list 
messages = [{
    "role": "system",
    "content": system_prompt
}, {
    "role": "user",
    "content": prompt 
}]

def display_summary_email(result):
    display(Markdown(result))

# Step 3: Call OpenAI
# response =

response = ollama.chat.completions.create(
    model="deepseek-r1:1.5b",
    messages=messages
)

# Step 4: print the result
# print(

result = response.choices[0].message.content

display_summary_email(result)

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success ⠋ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕███



**Subject: Your Application For... [Job Title] - [Company Name]**

**[Attention]**

Dear [Referred Party's Name],

I am writing to invite you to the exciting opportunity of applying for the position of [Job Title] at [Company Name]. With a passion for growth and teamwork, your application will play a crucial role in our successful journey.

Attached is my resume, which I kindly request your consideration. Your contribution as we proceed could make us even more distinguished. Thank you for taking the time to reach out.

Warm regards,

[Your Name]  
[Position Details or Contact Information]